# Detecting Rare Earth Elements (Neodymium) in Wyvern Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nrevyw/wyvern-public-resources/blob/main/tutorial-notebooks/detecting-rare-earth-elements/detecting_rare_earth_elements.ipynb)


This notebook is a lightweight, approachable walk-through of detecting the rare-earth
element **neodymium (Nd)** in Wyvern Dragonette hyperspectral imagery, using the
Mountain Pass REE mine in California as an example scene.

Nd(III) produces sharp, host-independent absorption features in the visible / near-infrared
(VNIR) at roughly **585, 745, 810 and 870 nm**. Because those positions barely shift with
the host mineral, they are a reliable spectral fingerprint for Nd-bearing material such as
the bastnaesite ore mined at Mountain Pass.

This tutorial is inspired by Asadzadeh, Koellner & Chabrillat (2024), *Detecting rare earth
elements using EnMAP hyperspectral satellite data: a case study from Mountain Pass,
California* (Scientific Reports 14:20766, https://doi.org/10.1038/s41598-024-71395-2). We
adapt the idea to Wyvern's Extended VNIR bands with three small, standard steps:

1. **Continuum removal** to isolate absorption features from overall brightness.
2. **Spectral resampling** of a real USGS reference spectrum (bastnaesite) onto Wyvern's bands.
3. **ACE** (Adaptive Cosine Estimator), a matched-filter-style detector, to map Nd-bearing pixels.

> **Note on data level:** REE absorptions are subtle, so this workflow needs **L2A surface
> reflectance** (atmospherically corrected). Running it on L1B top-of-atmosphere radiance
> mixes in the solar spectrum and the O2 absorption band near 760 nm and will not work.


## Setup

This notebook is self-contained and runs top to bottom in [Google Colab](https://colab.research.google.com/) or any local Jupyter install. Run the cell below first to install the packages it needs, then continue through the notebook. Nothing else needs to be set up.

In [ ]:
# Install the packages this notebook needs (Colab-ready).
# numpy, pandas, matplotlib, scipy and requests ship with Colab; we add
# rasterio (GeoTIFF I/O), Spectral Python (ACE detector) and pyarrow (Parquet).
%pip install -q rasterio spectral scipy pyarrow

In [ ]:
import io

import requests
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as warp_points
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy import ndimage
from spectral.algorithms.detectors import ace as spy_ace

## Configuration

The scene is Wyvern Dragonette-002 over Mountain Pass, available on the
[Wyvern Open Data Program](https://opendata.wyvern.space/#/?.language=en) as an L2A
surface-reflectance product. The reference spectrum comes from
[OpenSpecLib v0.0.6](https://github.com/null-jones/openspeclib/releases/tag/v0.0.6),
which packages the USGS splib07 library as Parquet.

In [ ]:
# Wyvern Open Data Program STAC item (L2A surface reflectance) for the Mountain Pass scene.
STAC_ITEM = (
    "https://wyvern-odp.com/application/mining/"
    "wyvern_dragonette-002_20250422T221452_2bc40137_l2a/"
    "wyvern_dragonette-002_20250422T221452_2bc40137_l2a.json"
)
DOWNLOAD_IMAGE = True  # set False after the first run

# L2A products store reflectance as scaled integers; multiply by this to get 0..1 reflectance.
REFLECTANCE_SCALE = 1e-4

# USGS splib07 reference spectrum: the pre-built Parquet asset published with the
# OpenSpecLib v0.0.6 release. We download it directly for convenience instead of
# installing openspeclib and converting splib07 ourselves.
OPENSPECLIB_PARQUET_URL = "https://github.com/null-jones/openspeclib/releases/download/v0.0.6/usgs_splib07.parquet"
BASTNAESITE_ID = "usgs_splib07:splib07a_Bastnaesite_REE_WS320_crystl_ASDFRb_AREF"

# Diagnostic Nd(III) VNIR absorption features, in nm.
ND_FEATURES = [585, 745, 810, 870]

# Area of interest: a bounding box around the Mountain Pass mine
# (min_lon, min_lat, max_lon, max_lat, WGS84). The ODP item is the full
# satellite swath, so we crop to this before analysis.
AOI_BOUNDS = (-115.555, 35.470, -115.515, 35.492)

# A pixel over the ore-handling area to inspect (lon, lat).
ORE_LON, ORE_LAT = -115.5241, 35.4793

## Download the scene from the Open Data Program

We read the STAC metadata, find the GeoTIFF asset, and stream it to disk.

In [ ]:
def geotiff_href(stac_item: dict) -> str:
    """Return the download URL of the GeoTIFF asset in a Wyvern STAC item.

    Args:
        stac_item: The parsed STAC item JSON.

    Returns:
        The href of the first asset that points to a GeoTIFF.
    """
    assets = stac_item["assets"]
    if "Cloud optimized GeoTiff" in assets:
        return assets["Cloud optimized GeoTiff"]["href"]
    for asset in assets.values():
        if str(asset.get("href", "")).lower().endswith((".tif", ".tiff")):
            return asset["href"]
    raise ValueError("No GeoTIFF asset found in the STAC item.")


stac_item = requests.get(STAC_ITEM, timeout=60)
stac_item.raise_for_status()
stac_item = stac_item.json()
download_url = geotiff_href(stac_item)
local_file = download_url.split("/")[-1]
print(f"STAC item: {stac_item['id']}")
print(f"GeoTIFF:   {local_file}")

if DOWNLOAD_IMAGE:
    with requests.get(download_url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(local_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 16):
                f.write(chunk)
    print("Download complete.")

## Load the cube, wavelengths and band widths

Wyvern GeoTIFFs describe each band as `Band_<wavelength>` and carry a per-band `FWHM`
(Full Width at Half Maximum) tag. We read the band centre wavelengths and FWHM values now,
because we will need them both to interpret the spectra and to resample the reference.

The ODP item covers the full satellite swath, so we crop to a bounding box around the mine (`AOI_BOUNDS`) and read only that window from the Cloud-Optimized GeoTIFF, which is both faster and far lighter on memory.

In [ ]:
def read_wavelengths_and_fwhm(
    dataset: rasterio.DatasetReader,
) -> tuple[np.ndarray, np.ndarray]:
    """Read band centre wavelengths and FWHM (both in nm) from a Wyvern GeoTIFF.

    Args:
        dataset: An open rasterio dataset.

    Returns:
        A tuple of the band centre wavelengths and the band FWHM values.
    """
    centres, fwhm = [], []
    for band in range(1, dataset.count + 1):
        centres.append(float(dataset.descriptions[band - 1].split("_")[1]))
        fwhm.append(float(dataset.tags(band)["FWHM"]))
    return np.asarray(centres), np.asarray(fwhm)


with rasterio.open(local_file) as src:
    wavelengths, fwhm = read_wavelengths_and_fwhm(src)
    # Reproject the AOI to the image CRS and read only that window. On a COG this
    # loads just the mine area, not the whole swath.
    left, bottom, right, top = transform_bounds("EPSG:4326", src.crs, *AOI_BOUNDS)
    window = from_bounds(left, bottom, right, top, src.transform)
    window = window.round_offsets().round_lengths()
    cube = src.read(window=window).astype("float32")
    cube[cube == src.nodata] = np.nan
    profile = src.profile
    profile.update(
        height=cube.shape[1],
        width=cube.shape[2],
        transform=src.window_transform(window),
    )

cube *= REFLECTANCE_SCALE

## Continuum removal

An absorption feature is easiest to measure relative to the local spectral background, or
"continuum". We remove the continuum by dividing each pixel's spectrum by its upper convex hull,
following the classic Clark & Roush (1984) approach ([Reflectance spectroscopy: Quantitative analysis techniques for remote sensing applications](https://doi.org/10.1029/JB089iB07p06329), J. Geophys. Res. 89, 6329-6340). The continuum is the convex hull fitted over the top of the spectrum (the background reflectance if the absorption were absent); dividing by it normalizes features to a common baseline so their depths and positions can be compared between pixels and against library spectra. Absorption features then appear as dips below 1.

In [ ]:
def continuum_removed(wavelengths: np.ndarray, reflectance: np.ndarray) -> np.ndarray:
    """Continuum-remove a spectrum by dividing by its upper convex hull.

    Args:
        wavelengths: Band centre wavelengths in nm.
        reflectance: Reflectance values at those wavelengths.

    Returns:
        The continuum-removed spectrum (1.0 = on the continuum, lower = absorption).
    """
    points = list(zip(wavelengths, reflectance))
    hull: list[tuple[float, float]] = []
    for p in points:
        while len(hull) >= 2:
            (x1, y1), (x2, y2) = hull[-2], hull[-1]
            # Pop while the last hull point is below the line from hull[-2] to p (keep upper hull).
            if (x2 - x1) * (p[1] - y1) - (y2 - y1) * (p[0] - x1) >= 0:
                hull.pop()
            else:
                break
        hull.append(p)
    hull_x = [h[0] for h in hull]
    hull_y = [h[1] for h in hull]
    return reflectance / np.interp(wavelengths, hull_x, hull_y)


def shade_features(ax: plt.Axes, label_y: float = -0.08) -> None:
    """Shade and label the diagnostic Nd feature windows on a plot.

    Args:
        ax: The Matplotlib axes to shade.
        label_y: Vertical position of the "Nd ###" labels in axes fraction;
            negative values place them below the x-axis tick labels.

    Returns:
        None. The axes are modified in place.
    """
    for centre in ND_FEATURES:
        ax.axvspan(centre - 10, centre + 10, color="gold", alpha=0.18, zorder=0)
        ax.text(
            centre,
            label_y,
            f"Nd {centre}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=7,
            color="#9a7d0a",
            clip_on=False,
        )


# Inspect a pixel over the ore-handling area, selected by its (lon, lat).
xs, ys = warp_points("EPSG:4326", profile["crs"], [ORE_LON], [ORE_LAT])
row, col = rowcol(profile["transform"], xs[0], ys[0])
pixel = cube[:, row, col]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
fig.suptitle(
    "Nd(III) absorption features in a Mountain Pass ore pixel (Dragonette-002 L2A)"
)
shade_features(ax[0], label_y=-0.115)
ax[0].plot(wavelengths, pixel, "o-", color="#1a5276", ms=3)
ax[0].set(title="Reflectance", xlabel="wavelength (nm)", ylabel="reflectance")
shade_features(ax[1], label_y=-0.115)
ax[1].plot(
    wavelengths, continuum_removed(wavelengths, pixel), "o-", color="#c0392b", ms=3
)
ax[1].axhline(1.0, color="grey", lw=0.6)
ax[1].set(title="Continuum-removed", xlabel="wavelength (nm)", ylabel="CR reflectance")
for a in ax:
    # Make room for the Nd labels between the tick labels and the axis title.
    a.xaxis.labelpad = 30
plt.tight_layout()
plt.show()

You should see clear dips at the shaded 585, 745 and 810 nm windows, the Nd(III)
fingerprint. (Dragonette's 815 nm band sits slightly off the true ~800 nm minimum, so that
feature reads a little shallow. This matters less for the full-spectrum ACE detector below:
it matches the shape of the whole spectrum at once, so no single band's depth decides the
result.)

## Load and resample the reference spectrum

Rather than hand-pick band positions, we compare each pixel to a real laboratory spectrum of **bastnaesite** (the Mountain Pass ore mineral) from the [USGS splib07 spectral library](https://dx.doi.org/10.3133/ds1035), accessed through the [OpenSpecLib v0.0.6](https://github.com/null-jones/openspeclib/releases/tag/v0.0.6) release, which repackages the splib07 release (a large collection of per-spectrum ASCII data files, alongside other public libraries) as Parquet for easy programmatic access; we download the pre-built Parquet file straight from the release rather than installing anything. We resample the reference onto Wyvern's bands by convolving with each band's response (a Gaussian approximated from the FWHM), so the reference is blurred to the same spectral resolution as the imagery.

In [ ]:
def load_reference(
    parquet_url: str, reference_id: str
) -> tuple[np.ndarray, np.ndarray]:
    """Load one USGS splib07 spectrum from an OpenSpecLib parquet file.

    Args:
        parquet_url: URL (or local path) of the usgs_splib07.parquet file.
        reference_id: The id of the spectrum to load.

    Returns:
        A tuple of wavelengths in nm and reflectance (0..1), with bad bands set to NaN.
    """
    if parquet_url.startswith("http"):
        raw = requests.get(parquet_url, timeout=600).content
        source: object = io.BytesIO(raw)
    else:
        source = parquet_url
    df = pd.read_parquet(source, columns=["id", "spectral_data.values"])
    values = np.asarray(
        df.loc[df["id"] == reference_id, "spectral_data.values"].iloc[0],
        dtype="float64",
    )
    values[(values < 0) | (values > 1.5)] = np.nan  # USGS bad-band flags
    # splib07a wavelength grid: 2151 samples, 350-2500 nm at 1 nm spacing
    # (measured on an ASD field spectroradiometer).
    wl_nm = np.linspace(0.34999999, 2.5, 2151) * 1000.0
    return wl_nm, values


def resample_to_bands(
    ref_wl: np.ndarray, ref_refl: np.ndarray, centres: np.ndarray, fwhm: np.ndarray
) -> np.ndarray:
    """Convolve a fine reference spectrum onto sensor bands using Gaussian responses.

    Args:
        ref_wl: Reference wavelengths in nm.
        ref_refl: Reference reflectance at those wavelengths.
        centres: Sensor band centre wavelengths in nm.
        fwhm: Sensor band FWHM in nm.

    Returns:
        The reference resampled to the sensor bands.

    Note:
        For simplicity this approximates each band's response as a Gaussian
        derived from its FWHM. A rigorous workflow would instead convolve with
        each band's actual relative spectral response (RSR) curve, which is
        published for Dragonette in the Wyvern public resources:
        https://github.com/Nrevyw/wyvern-public-resources/tree/main/relative-spectral-responses
    """
    good = np.isfinite(ref_refl)
    wl, refl = ref_wl[good], ref_refl[good]
    out = np.empty(len(centres))
    for i, (centre, width) in enumerate(zip(centres, fwhm)):
        # Gaussian approximation of the band response (see Note; real RSR curves
        # would be more faithful).
        weight = np.exp(-0.5 * ((wl - centre) / (width / 2.3548)) ** 2)
        out[i] = np.sum(weight * refl) / np.sum(weight)
    return out


ref_wl, ref_refl = load_reference(OPENSPECLIB_PARQUET_URL, BASTNAESITE_ID)
target = resample_to_bands(ref_wl, ref_refl, wavelengths, fwhm)

fig, ax = plt.subplots(figsize=(8, 4.2))
shade_features(ax)
m = (ref_wl >= 440) & (ref_wl <= 900)
ax.plot(
    ref_wl[m], ref_refl[m], color="#95a5a6", lw=1.3, label="USGS splib07 lab (1 nm)"
)
ax.plot(
    wavelengths,
    target,
    "s-",
    color="#1a5276",
    ms=4,
    lw=2,
    label="resampled to Dragonette bands",
)
ax.set(
    xlim=(440, 900),
    xlabel="wavelength (nm)",
    ylabel="reflectance",
    title="Bastnaesite reference (USGS splib07) resampled to Dragonette bands",
)
# Make room for the Nd labels between the tick labels and the axis title.
ax.xaxis.labelpad = 22
ax.legend()
plt.tight_layout()
plt.show()

## Detect with ACE

The Adaptive Cosine Estimator (ACE) is a standard sub-pixel target detector. It scores how well each pixel aligns with the reference direction after the scene background has been statistically whitened (mean-subtracted and decorrelated with the scene covariance). ACE is insensitive to overall brightness, which makes it a robust, easy-to-threshold detector. ACE is bounded between 0 and 1 where values closer to 1 are stronger matches to the reference spectra.

For a pixel spectrum $x$, target $t$, scene mean $m$ and covariance $C$, with centred vectors $\tilde{x} = x - m$ and $\tilde{t} = t - m$:

$$
\mathrm{ACE}(x) = \frac{\left(\tilde{t}^{\top} C^{-1} \tilde{x}\right)^2}{\left(\tilde{t}^{\top} C^{-1} \tilde{t}\right)\left(\tilde{x}^{\top} C^{-1} \tilde{x}\right)}
$$

We use the ACE implementation in [Spectral Python (SPy)](https://www.spectralpython.net/), a common hyperspectral package, and wrap it only to drop nodata pixels and reshape the result. Since we cropped to the mine, the scene is small, so this is fast and light.

In [ ]:
def detect_ace(cube: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Adaptive Cosine Estimator detection score over a reflectance cube.

    The detection is delegated to Spectral Python's ace; we only drop nodata
    pixels and reshape the result back onto the image grid.

    Args:
        cube: Reflectance cube (bands, rows, cols) with nodata set to NaN.
        target: Target reflectance vector (length equal to the number of bands).

    Returns:
        A 2D array of ACE scores in 0..1, with NaN where the cube is invalid.
    """
    bands, rows, cols = cube.shape
    flat = cube.reshape(bands, -1).T
    valid = np.isfinite(flat).all(axis=1)
    out = np.full(flat.shape[0], np.nan, dtype="float32")
    out[valid] = spy_ace(flat[valid], target)
    return out.reshape(rows, cols)


ace = detect_ace(cube, target)
print(
    f"ACE: median {np.nanmedian(ace):.3f}, "
    f"99th pct {np.nanpercentile(ace, 99):.3f}, "
    f"99.9th pct {np.nanpercentile(ace, 99.9):.3f}, max {np.nanmax(ace):.3f}"
)

## Map the result

We show a true-colour view next to the ACE score. Bright pixels are the strongest Nd matches.

In [ ]:
def stretch(nm_r: float, nm_g: float, nm_b: float) -> np.ndarray:
    """Build a 2nd-98th percentile stretched RGB composite from three wavelengths.

    Args:
        nm_r: Wavelength in nm for the red channel.
        nm_g: Wavelength in nm for the green channel.
        nm_b: Wavelength in nm for the blue channel.

    Returns:
        An (rows, cols, 3) array scaled to 0..1, ready for display.
    """

    def band(nm: float) -> np.ndarray:
        return cube[int(np.argmin(np.abs(wavelengths - nm)))]

    rgb = np.dstack([band(nm_r), band(nm_g), band(nm_b)])
    lo, hi = np.nanpercentile(rgb, 2), np.nanpercentile(rgb, 98)
    return np.clip((rgb - lo) / (hi - lo), 0, 1)


rgb = stretch(660, 550, 465)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(rgb)
ax[0].set_title("True-colour (660/550/465 nm)")
im = ax[1].imshow(ace, cmap="inferno", vmin=0, vmax=0.7)
ax[1].set_title("ACE (Bastnaesite target)")

# Give both panels an identical colorbar slot so the images stay the same size;
# only the ACE panel's slot actually holds a colorbar.
for axis, has_cbar in ((ax[0], False), (ax[1], True)):
    cax = make_axes_locatable(axis).append_axes("right", size="4%", pad=0.08)
    axis.axis("off")
    if has_cbar:
        fig.colorbar(im, cax=cax, label="ACE")
    else:
        cax.axis("off")
plt.tight_layout()
plt.show()

## Overlay the detections on the RGB preview

A detection map is most useful laid over a familiar view. We threshold the ACE score, apply a little morphological cleanup to drop isolated single-pixel hits and thicken the rest for visibility, then blend the result over the true-colour composite.

The 0.2 threshold is an empirical choice based on this scene's score distribution, not a magic number: background ACE scores cluster near zero (the scene median is ~0.01 and even the 99th percentile is ~0.18), so a 0.2 cutoff keeps only the extreme tail of strongest matches, which top out near 0.79. If you adapt this workflow to another scene or target, inspect the score statistics printed above and tune the threshold to balance missed detections against false positives.

In [ ]:
DETECTION_THRESHOLD = 0.20

mask = np.nan_to_num(ace) > DETECTION_THRESHOLD
# Additional processing: remove isolated single-pixel hits, then thicken slightly.
mask = ndimage.binary_opening(mask, structure=np.ones((2, 2)))
mask = ndimage.binary_dilation(mask, iterations=1)

rgb = stretch(660, 550, 465)
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(rgb)
overlay = np.where(mask, ace, np.nan)
im = ax.imshow(
    overlay, cmap="turbo", vmin=DETECTION_THRESHOLD, vmax=np.nanmax(ace), alpha=0.9
)
ax.set(title=f"Nd(III) detections (ACE > {DETECTION_THRESHOLD}) over RGB")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="ACE")
plt.tight_layout()
plt.show()

## Do the high-scoring pixels really look like the reference?

Always sanity-check a detector. We overlay the continuum-removed spectra of the top-scoring pixels against the reference; they should share the Nd absorptions.

In [ ]:
flat = cube.reshape(cube.shape[0], -1).T
scores = ace.ravel()
top = np.argsort(np.where(np.isfinite(scores), scores, -1))[::-1][:40]
top_mean = np.nanmean(flat[top], axis=0)

fig, ax = plt.subplots(figsize=(8, 4.2))
shade_features(ax)
for i in top[:20]:
    ax.plot(
        wavelengths,
        continuum_removed(wavelengths, flat[i]),
        color="#e59866",
        lw=0.6,
        alpha=0.5,
    )
ax.plot(
    wavelengths,
    continuum_removed(wavelengths, top_mean),
    "o-",
    color="#c0392b",
    ms=3,
    lw=2,
    label="mean of top-40 ACE pixels",
)
ax.plot(
    wavelengths,
    continuum_removed(wavelengths, target),
    "s-",
    color="#1a5276",
    ms=3,
    lw=2,
    label="Bastnaesite reference",
)
ax.set(
    xlabel="wavelength (nm)",
    ylabel="continuum-removed reflectance",
    title="High-ACE pixels vs Bastnaesite reference (continuum-removed)",
)
# Make room for the Nd labels between the tick labels and the axis title.
ax.xaxis.labelpad = 22
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

## Save the detection map

Write the ACE score to a georeferenced GeoTIFF you can open in QGIS, ArcGIS or ENVI.

In [ ]:
out_profile = profile.copy()
out_profile.update(count=1, dtype="float32", nodata=np.nan, compress="lzw")
out_profile.pop("photometric", None)
with rasterio.open("mountain_pass_nd_ace.tif", "w", **out_profile) as dst:
    dst.write(ace.astype("float32"), 1)
    dst.set_band_description(1, "nd_ace")
print("Wrote mountain_pass_nd_ace.tif")

## Wrap-up and caveats

You continuum-removed a Wyvern spectrum, resampled a real USGS bastnaesite reference to
Dragonette's bands, and used ACE to map Nd-bearing material at Mountain Pass, reproducing, in
miniature, the idea behind the EnMAP study.

A few things to keep in mind:

- **Use L2A, not L1B.** These features are shallow; top-of-atmosphere radiance will drown them
  in the solar/atmospheric signal (especially the O2 band near 760 nm).
- **It is an exploration screen, not proof of REE.** Treat bright pixels as candidates to follow
  up, and always sanity-check their spectra (step 9).
- **Spectral resolution matters.** Dragonette's ~20-30 nm bands broaden the narrow Nd features,
  so measured depths are smaller than a lab or a fine-resolution sensor like EnMAP would see.
  Requiring agreement across multiple features (as ACE effectively does) is what keeps it robust.

Questions or ideas? Join the [GitHub Discussions](https://github.com/Nrevyw/wyvern-public-resources/discussions).
